In [1]:
import sys
from pathlib import Path

AIMA_DIR = Path('/home/mai/SCI193611_ARTIFICIAL_INTELLIGENCE/aima')
if str(AIMA_DIR) not in sys.path:
    sys.path.insert(0, str(AIMA_DIR))

from agents import Agent, Thing, Direction, GraphicEnvironment


In [2]:
class Food(Thing):
    pass


class Water(Thing):
    pass


def snake_route(width, height):
    route = []
    for x in range(width):
        ys = range(height) if x % 2 == 0 else range(height - 1, -1, -1)
        route.extend((x, y) for y in ys)
    return route


In [3]:
class NoRepeatBlindDog(Agent):
    def __init__(self, route):
        super().__init__(program=self.program)
        self.route = route
        self.route_index = 0
        self.visited = {route[0]}
        self.direction = Direction('down')

    def moveforward(self):
        x, y = self.location
        if self.direction.direction == Direction.R:
            self.location = (x + 1, y)
        elif self.direction.direction == Direction.L:
            self.location = (x - 1, y)
        elif self.direction.direction == Direction.D:
            self.location = (x, y + 1)
        else:
            self.location = (x, y - 1)
        if self.location in self.visited:
            raise RuntimeError(f'พยายามเดินซ้ำช่อง {self.location}')
        self.visited.add(self.location)
        self.route_index += 1

    def eat(self, thing):
        return isinstance(thing, Food)

    def drink(self, thing):
        return isinstance(thing, Water)

    def program(self, percepts):
        for thing in percepts:
            if isinstance(thing, Food):
                return 'eat'
            if isinstance(thing, Water):
                return 'drink'

        if self.route_index >= len(self.route) - 1:
            return 'stop'

        target = self.route[self.route_index + 1]
        x, y = self.location
        tx, ty = target
        wanted = Direction.R if tx > x else Direction.L if tx < x else Direction.D if ty > y else Direction.U
        if self.direction.direction == wanted:
            return 'moveforward'
        right = self.direction + Direction.R
        return 'turnright' if right.direction == wanted else 'turnleft'


In [4]:
class Park2D(GraphicEnvironment):
    def percept(self, agent):
        return self.list_things_at(agent.location)

    def execute_action(self, agent, action):
        if action == 'turnright':
            agent.direction = agent.direction + Direction.R
        elif action == 'turnleft':
            agent.direction = agent.direction + Direction.L
        elif action == 'moveforward':
            old_location = agent.location
            agent.moveforward()
            print(f'เดินจาก {old_location} ไป {agent.location}')
        elif action == 'eat':
            food = self.list_things_at(agent.location, Food)[0]
            self.delete_thing(food)
            print(f'เก็บอาหารที่ {agent.location}')
        elif action == 'drink':
            water = self.list_things_at(agent.location, Water)[0]
            self.delete_thing(water)
            print(f'เก็บน้ำที่ {agent.location}')

    def is_done(self):
        return not any(isinstance(t, (Food, Water)) for t in self.things)


In [6]:
WIDTH, HEIGHT = 5, 5
route = snake_route(WIDTH, HEIGHT)
park = Park2D(WIDTH, HEIGHT, color={
    'NoRepeatBlindDog': (200, 0, 0),
    'Food': (230, 115, 40),
    'Water': (0, 200, 200),
})

dog = NoRepeatBlindDog(route)
park.add_thing(dog, route[0])
for location in [(1, 2), (4, 3)]:
    park.add_thing(Food(), location)
for location in [(0, 1), (2, 4)]:
    park.add_thing(Water(), location)

park.run(steps=100, delay=0.15)
print(f'ช่องที่เดินจริง: {len(dog.visited)} ช่อง, เดินซ้ำ: {len(dog.visited) != dog.route_index + 1}')
print('เส้นทางที่เดิน:', sorted(dog.visited))


,,,,
,,,,
,,,,
,,,,
,,,,


ช่องที่เดินจริง: 24 ช่อง, เดินซ้ำ: False
เส้นทางที่เดิน: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 0), (1, 1), (1, 2), (1, 3), (1, 4), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (3, 0), (3, 1), (3, 2), (3, 3), (3, 4), (4, 0), (4, 1), (4, 2), (4, 3)]
